In [ ]:
import pandas as pd

score_data = pd.read_csv("/nas/4t_copy/4t/cehou/LLM_safety/Stockholm/safety_ai.csv")
img_score = pd.read_csv("/nas/4t_copy/4t/cehou/LLM_safety/Stockholm/safety_score.csv")
image_id = pd.read_excel('/nas/4t_copy/4t/cehou/LLM_safety/Stockholm/FileIndex5000.xlsx')

score_data = score_data.drop(index=0).reset_index(drop=True)
score_data['Panel Member_Age'] = score_data['Panel Member_Age'].astype(int)

In [ ]:
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')
# 计算每个 ID 的 counts 的熵
def calculate_entropy(group):
    counts = group['counts'].values
    probabilities = counts / counts.sum()  # 归一化为概率分布
    return entropy(probabilities)

In [ ]:

def tidy_df(data, k):
    selected_data = data[['ExternalReferenceID.a.1', 'Safe_Unsafe', 'ImageID_A', 'ImageID_B', 'Panel Member_Gender', 'Panel Member_Age']]
    selected_data['Safe_Unsafe'] = selected_data['Safe_Unsafe'].apply(lambda x: 'Right' if x == 'Otrygg' else 'Left')
    selected_data['Panel Member_Gender'] = selected_data['Panel Member_Gender'].apply(lambda x: 'Male' if x == 'Man' else 'Female')
    selected_data_left = selected_data[['ExternalReferenceID.a.1', 'Safe_Unsafe', 'ImageID_A', 'Panel Member_Gender', 'Panel Member_Age']]
    selected_data_right = selected_data[['ExternalReferenceID.a.1', 'Safe_Unsafe', 'ImageID_B', 'Panel Member_Gender', 'Panel Member_Age']]
    selected_data_left['is_win'] = selected_data_left['Safe_Unsafe'].apply(lambda x: 1 if x == 'Left' else 0)
    selected_data_left = selected_data_left.rename(columns={'ImageID_A': 'ImageID'})
    selected_data_right['is_win'] = selected_data_right['Safe_Unsafe'].apply(lambda x: 1 if x == 'Right' else 0)
    selected_data_right = selected_data_right.rename(columns={'ImageID_B': 'ImageID'})
    select_data_total = pd.concat([selected_data_left, selected_data_right])
    select_data_total['counts'] = select_data_total['ImageID'].map(select_data_total['ImageID'].value_counts())
    select_data_total = select_data_total.rename(columns={'ImageID': 'ID'})
    select_data_total['ID'] = select_data_total['ID'].astype(int)

    select_data_total_score = select_data_total.merge(img_score[['ID','Score']], on='ID')
    select_df = select_data_total_score[select_data_total_score['counts'] > k].reset_index(drop=True)
    select_df['Score_Group'] = pd.cut(select_df['Score'], bins=range(1, 11), right=False, labels=[f"[{i}, {i+1})" for i in range(1, 10)])
    # 按 ID 分组并计算熵
    id_entropy = select_df.groupby('ID').apply(calculate_entropy).reset_index()
    id_entropy.columns = ['ID', 'var']

    # 将 id_entropy 合并到 select_data_total_score 中
    select_df = select_df.merge(id_entropy, on='ID', how='left')
    output_df = select_df[['ID', 'Score', 'var', 'Score_Group']].drop_duplicates().reset_index(drop=True)
    return output_df

data = score_data
all_output_df = tidy_df(data, 1)

man_data = score_data[score_data['Panel Member_Gender'] == 'Man'].reset_index(drop=True)
man_output_df = tidy_df(man_data, 1)
man_output_df = man_output_df.rename(columns={
    'Score': 'Score_man',
    'var': 'var_man'
})
female_data = score_data[score_data['Panel Member_Gender'] == 'Kvinna'].reset_index(drop=True)
female_output_df = tidy_df(female_data, 1)
female_output_df = female_output_df.rename(columns={
    'Score': 'Score_female',
    'var': 'var_female'
})

elder_data = score_data[score_data['Panel Member_Age'] > 44].reset_index(drop=True)
elder_output_df = tidy_df(elder_data, 1)
elder_output_df = elder_output_df.rename(columns={
    'Score': 'Score_elder',
    'var': 'var_elder'
})

young_data = score_data[score_data['Panel Member_Age'] <= 44].reset_index(drop=True)
young_output_df = tidy_df(young_data, 1)
young_output_df = young_output_df.rename(columns={
    'Score': 'Score_young',
    'var': 'var_young'
})

In [ ]:
plot_df = all_output_df.merge(man_output_df.iloc[:,:-1], on='ID', how='left').merge(
    female_output_df.iloc[:,:-1], on='ID', how='left').merge(
    young_output_df.iloc[:,:-1], on='ID', how='left').merge(
    elder_output_df.iloc[:,:-1], on='ID', how='left')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ==========================================
# 修复字体报错的关键代码
# ==========================================
# 设置字体系列为 sans-serif (无衬线字体)
plt.rcParams['font.family'] = 'sans-serif'
# 设置具体的字体查找顺序：
# 优先找 'Arial' -> 找不到就找 'DejaVu Sans' (Matplotlib默认) -> 最后用系统默认
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Bitstream Vera Sans', 'sans-serif']

# ==========================================
# 数据处理 (保持不变)
# ==========================================
# 假设 plot_df 已经在你的环境中定义好了
cols_to_plot = ['var', 'var_man', 'var_female', 'var_young', 'var_elder']

df_long = plot_df.melt(
    id_vars=['Score_Group'], 
    value_vars=cols_to_plot,
    var_name='Category',   
    value_name='Variance'
)

label_mapping = {
    'var': 'Overall',
    'var_man': 'Male',
    'var_female': 'Female',
    'var_young': 'Young',
    'var_elder': 'Elder'
}
df_long['Category'] = df_long['Category'].map(label_mapping)

# 排序
group_order = sorted(plot_df['Score_Group'].dropna().unique()) 

# ==========================================
# 绘图
# ==========================================
plt.figure(figsize=(12, 6))

# 使用默认主题，但根据上面的 rcParams 自动调整字体
sns.set_theme(style="whitegrid", rc={'font.family': 'sans-serif'}) 

sns.boxplot(
    data=df_long,
    x='Score_Group',
    y='Variance',
    hue='Category',
    order=group_order,
    palette="Set2",
    width=0.7,
    linewidth=1.5
)

# ==========================================
# 关键设置: Y 轴从 0 开始
# ==========================================
plt.ylim(bottom=0.5)

plt.title('Variance Distribution by Score Group and Demographics', fontsize=14)
plt.ylabel('Variance (Safety Perception)', fontsize=12)
plt.xlabel('Score Group', fontsize=12)
plt.legend(title='Demographic Group')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ==========================================
# 1. 字体与环境设置
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans', 'sans-serif']

my_palette = {
    'Overall': '#95a5a6', # 沉稳的灰色 (Concrete Grey)
    'Male':    '#3498db', # 蓝色 (Blue)
    'Female':  '#e74c3c', # 红色 (Red)
    'Young':   '#3498db', # 蓝色 (复用，保持左右视觉一致性)
    'Elder':   '#e74c3c'  # 红色 (复用)
}

# ==========================================
# 2. 数据准备 (保持不变)
# ==========================================
cols_gender = ['var', 'var_man', 'var_female']
cols_age    = ['var', 'var_young', 'var_elder']

# 准备左图数据
df_gender = plot_df.melt(
    id_vars=['Score_Group'], 
    value_vars=cols_gender,
    var_name='Category', value_name='Variance'
)

# 准备右图数据
df_age = plot_df.melt(
    id_vars=['Score_Group'], 
    value_vars=cols_age,
    var_name='Category', value_name='Variance'
)

# 标签映射
label_mapping = {
    'var': 'Overall', 'var_man': 'Male', 'var_female': 'Female',
    'var_young': 'Young', 'var_elder': 'Elder'
}
df_gender['Category'] = df_gender['Category'].map(label_mapping)
df_age['Category'] = df_age['Category'].map(label_mapping)

group_order = sorted(plot_df['Score_Group'].dropna().unique())

# ==========================================
# 3. 绘图 (修改了 Legend 位置)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
sns.set_theme(style="whitegrid", rc={'font.family': 'sans-serif'})

# --- 左图：性别对比 ---
sns.boxplot(
    data=df_gender,
    x='Score_Group', y='Variance', hue='Category',
    order=group_order,
    palette=my_palette, 
    ax=axes[0],
    width=0.6, linewidth=1.5
)
axes[0].set_title('Gender Group vs. Overall', fontsize=14, pad=10)
axes[0].set_ylabel('Variance of Safety Perception', fontsize=12)
axes[0].set_xlabel('Score Group', fontsize=12)
axes[0].set_ylim(bottom=0.5)

legend_patches_left = [
    mpatches.Patch(color='#95a5a6', label='Overall'),
    mpatches.Patch(color='#3498db', label='Male'),
    mpatches.Patch(color='#e74c3c', label='Female')
]

# 【修改点】强制图例在左上角
axes[0].legend(title='Group', handles=legend_patches_left, loc='upper right') 


# --- 右图：年龄对比 ---
sns.boxplot(
    data=df_age,
    x='Score_Group', y='Variance', hue='Category',
    order=group_order,
    palette=my_palette, 
    ax=axes[1],
    width=0.6, linewidth=1.5
)
axes[1].set_title('Age Group vs. Overall', fontsize=14, pad=10)
axes[1].set_xlabel('Score Group', fontsize=12)
axes[1].set_ylabel('')

legend_patches = [
    mpatches.Patch(color='#95a5a6', label='Overall'),      # 灰色 (对应 Overall)
    mpatches.Patch(color='#3498db', label='Middle-aged'),  # 蓝色 (对应原来的 Young)
    mpatches.Patch(color='#e74c3c', label='Elderly')       # 红色 (对应原来的 Elder)
]

# 【修改点】强制图例在左上角
axes[1].legend(title='Group', handles=legend_patches, loc='upper right')


plt.tight_layout()
plt.savefig("safety_perception_variance_comparison.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ... (你的数据处理代码保持不变) ...

# ==========================================
# 优化配色：灰色基准 + 冷暖对比
# ==========================================
# 这种配色方案通常被称为 "Highlight" 风格
my_palette = {
    'Overall': '#95a5a6', # 沉稳的灰色 (Concrete Grey)
    'Male':    '#3498db', # 蓝色 (Blue)
    'Female':  '#e74c3c', # 红色 (Red)
    'Young':   '#3498db', # 蓝色 (复用，保持左右视觉一致性)
    'Elder':   '#e74c3c'  # 红色 (复用)
}

# 绘图设置
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
sns.set_theme(style="whitegrid", rc={'font.family': 'sans-serif'})

# 左图
sns.boxplot(
    data=df_gender,
    x='Score_Group', y='Variance', hue='Category',
    order=group_order,
    palette=my_palette,  # <--- 使用自定义配色
    ax=axes[0],
    width=0.6, linewidth=1.2, fliersize=3
)
axes[0].set_title('Gender Group vs. Overall', fontsize=14)
axes[0].legend(title='', loc='upper right') # 去掉 title 让图例更干净

# 右图
sns.boxplot(
    data=df_age,
    x='Score_Group', y='Variance', hue='Category',
    order=group_order,
    palette=my_palette,  # <--- 使用自定义配色
    ax=axes[1],
    width=0.6, linewidth=1.2, fliersize=3
)
axes[1].set_title('Age Group vs. Overall', fontsize=14)
axes[1].set_xlabel('Score Group', fontsize=12)
axes[1].set_ylabel('')
axes[1].legend(title='', loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit

# ==========================================
# 1. 生成模拟数据 (模拟你的真实数据结构)
# ==========================================
np.random.seed(42)
n_scenes = 200  # 假设有200个不同的建成环境场景
n_reviews_per_scene = 50 # 每个场景有50个人打分

# 生成场景的"真实"内在质量 (0到10分)
true_quality = np.random.uniform(0, 10, n_scenes)

# 关键假设模拟：
# 如果分数接近0或10（极端），噪声(方差)小；如果分数接近5（模糊），噪声(方差)大。
# 我们用一个高斯分布来模拟打分，sigma(标准差)随着质量变化。
# Sigma function: 倒抛物线，中间大两头小
noise_level = -0.04 * (true_quality - 5)**2 + 1.5 
noise_level = np.clip(noise_level, 0.5, 2.0) # 限制最小和最大噪声

data = []
for i in range(n_scenes):
    # 模拟不同人群的打分
    # 假设不同性别对同一场景的基础感知是一致的，但在模糊场景下波动不同
    scene_id = f"Scene_{i}"
    base_score = true_quality[i]
    sigma = noise_level[i]
    
    for user_id in range(n_reviews_per_scene):
        # 随机分配性别和年龄
        gender = np.random.choice(['Male', 'Female'])
        age_group = np.random.choice(['Young', 'Old'])
        
        # 生成评分 (加入随机噪声)
        score = np.random.normal(base_score, sigma)
        score = np.clip(score, 0, 10) # 截断在0-10之间
        
        data.append({
            'Scene_ID': scene_id,
            'Gender': gender,
            'Age_Group': age_group,
            'Score': score
        })

df = pd.DataFrame(data)

# ==========================================
# 2. 分析步骤 A: 验证"倒U型"方差分布
# ==========================================

# 按场景聚合，计算均值(Mean)和标准差(Std)
scene_stats = df.groupby('Scene_ID')['Score'].agg(['mean', 'std']).reset_index()

# 拟合二次曲线 (Parabola fitting) 来证明倒U型关系
# y = ax^2 + bx + c
z = np.polyfit(scene_stats['mean'], scene_stats['std'], 2)
p = np.poly1d(z)
xp = np.linspace(0, 10, 100)

plt.figure(figsize=(12, 5))

# 子图1: 均值 vs 标准差
plt.subplot(1, 2, 1)
sns.scatterplot(data=scene_stats, x='mean', y='std', alpha=0.6, color='blue', label='Data Points')
plt.plot(xp, p(xp), 'r--', lw=3, label='Quadratic Fit (Trend)')
plt.title('Evidence 1: Consensus in Extremes\n(Mean Score vs. Standard Deviation)')
plt.xlabel('General Safety Perception (Mean Score)')
plt.ylabel('Divergence (Standard Deviation)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# ==========================================
# 3. 分析步骤 B: 验证"可迁移性" (群组间相关性)
# ==========================================

# 挑选出"高置信度"样本 (方差最低的25%的场景) vs "模糊"样本 (方差最高的25%)
threshold_low = scene_stats['std'].quantile(0.25)
threshold_high = scene_stats['std'].quantile(0.75)

high_confidence_scenes = scene_stats[scene_stats['std'] <= threshold_low]['Scene_ID']
ambiguous_scenes = scene_stats[scene_stats['std'] >= threshold_high]['Scene_ID']

# 准备群组对比数据 (Male vs Female)
pivot_gender = df.pivot_table(index='Scene_ID', columns='Gender', values='Score', aggfunc='mean')

# 计算相关性
corr_overall = pivot_gender['Male'].corr(pivot_gender['Female'])
corr_high_conf = pivot_gender.loc[pivot_gender.index.isin(high_confidence_scenes), 'Male'].corr(pivot_gender.loc[pivot_gender.index.isin(high_confidence_scenes), 'Female'])
corr_ambiguous = pivot_gender.loc[pivot_gender.index.isin(ambiguous_scenes), 'Male'].corr(pivot_gender.loc[pivot_gender.index.isin(ambiguous_scenes), 'Female'])

# 子图2: 不同子集下的群组相关性
plt.subplot(1, 2, 2)
corrs = [corr_overall, corr_ambiguous, corr_high_conf]
labels = ['Overall Data', 'Ambiguous Scenes\n(High Variance)', 'Distinct Visual Cues\n(Low Variance)']
colors = ['grey', 'red', 'green']

bars = plt.bar(labels, corrs, color=colors, alpha=0.8)
plt.title('Evidence 2: Knowledge Transfer Feasibility\n(Male-Female Score Correlation)')
plt.ylabel('Pearson Correlation Coefficient')
plt.ylim(0, 1.1)

# 在柱状图上标数值
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, round(yval, 3), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('safety_perception_analysis.png')